# 03 — Validación, comparación y sensibilidad

Validación cruzada estratificada de tres pliegues sobre los mismos 12,000 ejemplos para ambos modelos. Luego se mueven L2, pasos de propagación y tamaño de entrenamiento.

In [1]:
from pathlib import Path
import sys, urllib.request
import numpy as np

SEED = 2026
rng = np.random.default_rng(SEED)

# Funciona desde notebooks/ en el repositorio y también en Colab.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "raw" / "mnist.npz"
SRC = ROOT / "src"
if not SRC.exists():
    # Si se subió solo el notebook, crea una ruta local y descarga los datos.
    ROOT = Path.cwd()
    DATA = ROOT / "mnist.npz"
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", DATA
    )
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

data = np.load(DATA)
x_train = data["x_train"].astype(np.float32) / 255.0
y_train = data["y_train"].astype(np.int64)
x_test = data["x_test"].astype(np.float32) / 255.0
y_test = data["y_test"].astype(np.int64)
print("Semilla:", SEED, "| entrenamiento:", x_train.shape, "| prueba:", x_test.shape)

Semilla: 2026 | entrenamiento: (60000, 28, 28) | prueba: (10000, 28, 28)


In [2]:
from modelado_mnist import *
raw = raw_features(x_train)
graph = graph_message_features(x_train, alpha=0.35, steps=2)
idx = stratified_indices(y_train, 12000, seed=SEED)
y = y_train[idx]
rows = []
for fold, (tr, va) in enumerate(stratified_folds(y, 3, SEED), 1):
    for name, feats in (("Softmax", raw[idx]), ("Graph-MLP", graph[idx])):
        model = SoftmaxRegression(784, seed=SEED+fold) if name == "Softmax" else GraphMLP(784, seed=SEED+fold)
        model.fit(feats[tr], y[tr], feats[va], y[va], epochs=7, seed=SEED+fold)
        m = classification_metrics(y[va], model.predict_proba(feats[va]))
        rows.append((fold, name, m["accuracy"], m["f1_macro"], m["log_loss"]))
rows

[(1, 'Softmax', 0.888, 0.8876950619966613, 0.40487486124038696),
 (1, 'Graph-MLP', 0.91, 0.9097825411988684, 0.308861643075943),
 (2, 'Softmax', 0.892, 0.8913062759549714, 0.402599573135376),
 (2, 'Graph-MLP', 0.91275, 0.9123133538781696, 0.311621755361557),
 (3, 'Softmax', 0.89175, 0.8916043275051269, 0.41084104776382446),
 (3, 'Graph-MLP', 0.91775, 0.9174911729544302, 0.3035197854042053)]

In [3]:
for name in ("Softmax", "Graph-MLP"):
    r = np.array([[a, f, l] for _, n, a, f, l in rows if n == name])
    print(name, "media [accuracy, F1, log-loss] =", r.mean(axis=0).round(4), "DE =", r.std(axis=0, ddof=1).round(4))

Softmax media [accuracy, F1, log-loss] = [0.8906 0.8902 0.4061] DE = [0.0022 0.0022 0.0043]
Graph-MLP media [accuracy, F1, log-loss] = [0.9135 0.9132 0.308 ] DE = [0.0039 0.0039 0.0041]


## Sensibilidad reproducida desde el experimento completo

In [4]:
import json
results_path = ROOT / "data" / "processed" / "resultados.json"
if results_path.exists():
    results = json.loads(results_path.read_text(encoding="utf-8"))
    for row in results["sensitivity"]:
        print(f"{row['assumption']:24s} {str(row['value']):>8s} | F1={row['f1_macro']:.4f}")
else:
    print("Ejecute src/ejecutar_experimentos.py para regenerar la tabla completa.")

Regularizacion L2             0.0 | F1=0.9251
Regularizacion L2          0.0001 | F1=0.9251
Regularizacion L2           0.001 | F1=0.9237
Pasos de propagacion            0 | F1=0.9273
Pasos de propagacion            1 | F1=0.9256
Pasos de propagacion            2 | F1=0.9251
Pasos de propagacion            3 | F1=0.9221
Tamano de entrenamiento      5000 | F1=0.8882
Tamano de entrenamiento     10000 | F1=0.9216
Tamano de entrenamiento     15000 | F1=0.9259


## Interpretación

El resultado es estable ante L2 entre 0 y 10⁻⁴. Tres pasos reducen el F1 por sobre-suavizado: los nodos vecinos se vuelven demasiado parecidos. La conclusión es más sensible al tamaño de entrenamiento; con 5,000 ejemplos el F1 cae cerca de cuatro puntos. La ablation con cero pasos también revela que la no linealidad explica una parte mayor de la mejora que la propagación fija, por lo que no se atribuye toda la ganancia al grafo.